# scikit-learn과 예측 Pipeline

## Bike Sharing 수요 예측 실험

시간별 대여수요를 target으로 정하고 전처리와 모형을 하나의 Pipeline으로 묶어 baseline과 후보모형을 비교합니다.


## 이번 실습에서 완성할 것

- 예측대상·입력변수·제외열을 구분합니다.
- 같은 행의 전처리 전후 숫자표를 읽습니다.
- 학습자료 내부 검증으로 tree 깊이와 시간변수 포함 여부를 비교합니다.

**완료 결과:** 입력변수 역할표, 기준모형과 후보 MAE 표, validation 비교문장


## 이번 교시에서 사용할 데이터

### 강의 시연: UCI Steel Industry Energy Consumption

| 항목 | 내용 |
|---|---|
| 수집 맥락 | 한국 철강산업의 2018년 전력·운전 기록 |
| 규모 | 35,040행, 15분 간격 1년 자료 |
| 한 행의 의미 | 특정 15분 구간의 전력사용과 운전조건 |
| target | `Usage_kWh` |
| 주요 feature | 무효전력, power factor, `NSM`, `WeekStatus`, `Day_of_week`, `Load_Type` |

`date`는 시간정보이고 `NSM`은 자정 이후 경과 초입니다. `CO2(tCO2)`는 전력사용량과 매우 밀접한 결과 proxy일 수 있으므로 예측시점과 연구목적을 확인한 뒤 제외합니다. 5교시에는 시간순 미래예측이 아니라 같은 기간의 관측을 무작위로 나눈 holdout에서 Pipeline의 구조를 확인합니다.

### 적용 실습: UCI Bike Sharing

2011~2012년의 17,379개 시간별 관측에서 `cnt`를 예측합니다. 날씨·달력·시간 변수를 feature로 사용하고, `cnt`를 구성하는 `casual`과 `registered`는 제외합니다. Steel과 Bike의 분야는 다르지만 `target–feature–split–preprocessing–baseline–metric` 명세는 같은 방식으로 작성합니다.


## 이번 실습의 분석 순서

예측대상·입력변수·제외열을 먼저 구분합니다. fit에서 전처리 규칙과 모형을 학습하고 validation에서 세 후보를 비교한 뒤, 선택된 Pipeline 하나만 test에서 마지막으로 평가합니다.

**남길 결과:** 열 역할표, fit·validation·test 요약, validation MAE 후보표, 선택모형의 최종 test 표


## 실습 구간 안내

| 구분 | 수행 범위 |
|---|---|
| **수업 중 필수** | 열 역할, fit 전처리, validation 후보 비교, 최종 test·PASS |
| **시간이 남으면** | 시간대별 validation 예측과 tree 깊이 |
| **수업 후 확장(선택)** | `hr` 포함 여부의 추가 예측가치 |

수업 중에는 필수 구간의 결과와 마지막 `PASS`를 먼저 완성합니다. 확장 구간을 실행하지 않아도 필수 셀이 독립적으로 작동하도록 구성되어 있습니다.


## 실행 환경


In [ ]:
from pathlib import Path
import platform
import warnings

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 130)
warnings.filterwarnings("ignore", category=FutureWarning)

def find_course_data(relative_path: str) -> Path:
    """0교시에서 준비한 data 폴더의 파일을 찾습니다."""
    relative = Path(relative_path)
    checked = []
    for start in [Path.cwd(), *Path.cwd().parents, Path("/content")]:
        candidate = (start / "data" / relative).resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if candidate.exists():
            return candidate
    raise FileNotFoundError(
        f"데이터 파일을 찾지 못했습니다: {relative_path}. "
        "과정 폴더의 00_setup.ipynb를 먼저 실행해 주세요."
    )
print("Python:", platform.python_version())
print("pandas:", pd.__version__)


## 1. 변수 역할과 예측 시점

target, 사용할 feature, 제외할 결과·미래 정보를 명시합니다. 숫자·범주 열은 서로 다른 전처리를 사용합니다.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeRegressor

DATA_PATH = find_course_data("bike/hour.csv")
data = pd.read_csv(DATA_PATH, parse_dates=["dteday"])
analysis_name = "Bike demand regression"
target_name = "cnt"
numeric_features = ["temp", "atemp", "hum", "windspeed"]
categorical_features = ["season", "yr", "mnth", "hr", "holiday", "weekday", "workingday", "weathersit"]
feature_columns = numeric_features + categorical_features

print("shape:", data.shape)
print("target:", target_name)
print("excluded target components: casual, registered")
display(data[[target_name, *feature_columns]].head(3))
role_table = pd.DataFrame({
    "역할": ["target", "수치형 feature", "범주형 feature", "누수 때문에 제외"],
    "열": [target_name, ", ".join(numeric_features), ", ".join(categorical_features), "casual, registered"],
})
print("\n[모델에 넣기 전에 열의 역할 구분]")
display(role_table)


## 2. 학습·검증·평가자료와 전처리 규칙

전체 자료에서 test를 먼저 떼어 마지막 평가용으로 보관합니다. 남은 train을 fit과 validation으로 다시 나눕니다.

- fit: 결측 대체값·표준화 기준·범주 목록과 모형 규칙 학습
- validation: 평균 기준모형·선형회귀·Decision Tree 비교와 깊이 설정
- test: 선택된 Pipeline 하나의 최종 MAE를 한 번 계산

이번 교시는 API 구조를 익히기 위해 무작위 분할을 사용합니다. 미래기간 예측은 7교시에서 시간순으로 검증합니다. 다음 코드에서 전처리 미리보기와 후보모형은 fit 자료로만 규칙을 배우고 validation에는 그 규칙을 적용합니다.


In [ ]:
X = data[feature_columns]
y = data[target_name]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)
X_fit, X_validation, y_fit, y_validation = train_test_split(
    X_train, y_train, test_size=0.20, random_state=43
)

preprocess = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ]), numeric_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), categorical_features),
])

print("fit/validation/test:", X_fit.shape, X_validation.shape, X_test.shape)
print("y fit/validation/test:", y_fit.shape, y_validation.shape, y_test.shape)
split_view = pd.DataFrame({
    "구분": ["fit", "validation", "test"],
    "행 수": [len(X_fit), len(X_validation), len(X_test)],
    "target 평균": [y_fit.mean(), y_validation.mean(), y_test.mean()],
    "target 최솟값": [y_fit.min(), y_validation.min(), y_test.min()],
    "target 최댓값": [y_fit.max(), y_validation.max(), y_test.max()],
})
print("\n[학습용과 평가용 자료]")
display(split_view.round(2))
print("전처리는 fit에서 학습되고 같은 규칙이 validation과 test에 적용됩니다.")

from sklearn.base import clone
preview_preprocess = clone(preprocess)
preview_matrix = preview_preprocess.fit_transform(X_fit)
preview_columns = preview_preprocess.get_feature_names_out()
transformed_fit = pd.DataFrame(preview_matrix, index=X_fit.index, columns=preview_columns)

preview_numeric = numeric_features[0]
preview_category = categorical_features[-1]
encoded_category_columns = [
    column for column in preview_columns
    if column.startswith(f"cat__{preview_category}_")
]
print("\n[전처리 전: 원자료의 숫자열과 범주열]")
display(X_fit.loc[X_fit.head(5).index, [preview_numeric, preview_category]])
print("[전처리 후: 같은 행의 모델 입력용 숫자표]")
display(
    transformed_fit.loc[
        X_fit.head(5).index,
        [f"num__{preview_numeric}", *encoded_category_columns],
    ].round(3)
)

numeric_medians = pd.DataFrame({
    "수치형 feature": numeric_features,
    "fit에서 배운 중앙값": preview_preprocess
        .named_transformers_["num"]
        .named_steps["imputer"]
        .statistics_,
})
print("[빈 값이 생겼을 때 적용할 train 중앙값]")
display(numeric_medians.round(3))


## 3. 평균 기준모형과 후보모형 비교

모든 후보를 동일한 split과 metric에서 비교합니다. Pipeline은 전처리를 train 안에서 학습하게 합니다.


In [ ]:
models = {
    "mean baseline": DummyRegressor(strategy="mean"),
    "linear regression": Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", LinearRegression()),
    ]),
    "decision tree": Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", DecisionTreeRegressor(
            max_depth=8,
            min_samples_leaf=10,
            random_state=42,
        )),
    ]),
}

# 후보와 설정 비교에는 train 내부 validation만 사용합니다.
selection_rows = []
fitted_candidates = {}
validation_predictions = {}
for name, model in models.items():
    candidate = clone(model)
    candidate.fit(X_fit, y_fit)
    prediction = candidate.predict(X_validation)
    selection_rows.append({
        "model": name,
        "validation_mae": mean_absolute_error(y_validation, prediction),
    })
    fitted_candidates[name] = candidate
    validation_predictions[name] = prediction

validation_table = (
    pd.DataFrame(selection_rows)
    .sort_values("validation_mae")
    .reset_index(drop=True)
)
selected_model_name = validation_table.loc[0, "model"]

baseline_model = fitted_candidates["mean baseline"]
baseline_value = float(np.asarray(baseline_model.constant_).reshape(-1)[0])
baseline_prediction = validation_predictions["mean baseline"]
linear_prediction = validation_predictions["linear regression"]
tree_prediction = validation_predictions["decision tree"]

print("[train 내부 validation에서 세 기본 후보 비교]")
display(validation_table.round(3))
print("현재 기본 후보 중 validation MAE 선두:", selected_model_name)
print("아직 test는 예측하지 않았습니다. depth와 hr 비교를 마친 뒤 마지막 절에서 한 번 평가합니다.")

# 뒤의 검토 셀이 사용하는 호환 이름입니다. 이 표에는 test 점수가 없습니다.
result_table = validation_table.copy()


## 시간이 남으면 — 시간대별 예측과 tree 깊이 비교

validation의 시간대별 실제 평균과 세 후보의 평균예측을 비교해 실패 조건을 살펴봅니다. Tree 깊이는 test를 다시 보지 않고, 원래 train 안에서 나눈 fit과 validation으로 비교합니다.

### 실행 전 2분

1. 8시와 18시 가운데 어느 시간대에서 평균 기준모형의 오차가 더 클지 예상합니다.
2. 깊이 8을 4로 줄이면 fit과 validation 가운데 어느 MAE가 더 크게 변할지 예상합니다.

### 실행 후 6분

1. 그림에서 실제 수요와 예측 차이가 큰 시간대를 하나 고릅니다.
2. `DEPTH_TO_TRY`를 3, 4 또는 6 중 하나로 바꾸고 셀을 다시 실행합니다.
3. 변경 전후 fit MAE·validation MAE·두 값의 격차를 기록합니다.

### 짝과 확인 2분

선형회귀는 어떤 식을, tree는 어떤 분기규칙을 사용했는지 결과표의 한 행을 예로 들어 설명합니다.


In [ ]:
all_prediction_view = X_validation[["hr"]].copy()
all_prediction_view["actual"] = y_validation
all_prediction_view["mean_baseline"] = baseline_prediction
all_prediction_view["linear_regression"] = linear_prediction
all_prediction_view["decision_tree_depth_8"] = tree_prediction

hourly_prediction = all_prediction_view.groupby("hr").mean(numeric_only=True)
print("[validation에서 시간대별 실제 평균과 모형별 평균예측]")
display(hourly_prediction.loc[[0, 8, 12, 18, 23]].round(1))

import matplotlib.pyplot as plt
hourly_prediction.plot(figsize=(9, 4), marker="o", title="Actual and predicted mean demand by hour")
plt.ylabel("Bike rentals/hour")
plt.tight_layout()
plt.show()

from sklearn.base import clone

X_depth_fit, X_depth_validation, y_depth_fit, y_depth_validation = train_test_split(
    X_train, y_train, test_size=0.20, random_state=43
)
DEPTH_TO_TRY = 4  # 3, 4, 6 가운데 하나로 바꾸어 다시 실행해 보세요.

depth_rows = []
for depth in [8, DEPTH_TO_TRY]:
    candidate_tree = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", DecisionTreeRegressor(
            max_depth=depth,
            min_samples_leaf=10,
            random_state=42,
        )),
    ])
    candidate_tree.fit(X_depth_fit, y_depth_fit)
    fit_prediction = candidate_tree.predict(X_depth_fit)
    validation_prediction = candidate_tree.predict(X_depth_validation)
    depth_rows.append({
        "max_depth": depth,
        "fit_MAE": mean_absolute_error(y_depth_fit, fit_prediction),
        "validation_MAE": mean_absolute_error(y_depth_validation, validation_prediction),
    })

depth_comparison = pd.DataFrame(depth_rows)
depth_comparison["validation_minus_fit_MAE"] = (
    depth_comparison["validation_MAE"] - depth_comparison["fit_MAE"]
)
print("[test를 사용하지 않고 train 내부에서 tree 깊이만 변경]")
print("fit/validation 행·전처리·MAE·seed는 같습니다.")
display(depth_comparison.round(3))


## 수업 후 확장(선택) — `hr` 정보의 추가 예측가치

자전거 수요는 시간대에 따라 달라집니다. 같은 fit·validation 행과 Linear Regression을 유지하고 `hr`을 제외한 입력과 포함한 입력의 validation MAE를 비교합니다. 이 실험에서도 test는 사용하지 않습니다.

결과를 보기 전에 `hr`을 추가하면 validation MAE가 줄어들지 예상하고, 실행 후에는 차이를 건/시간 단위로 기록합니다.


In [ ]:
feature_set_rows = []
for feature_set_name, use_hour in [("without_hr", False), ("with_hr", True)]:
    selected_numeric = list(numeric_features)
    selected_categorical = [
        column for column in categorical_features
        if use_hour or column != "hr"
    ]
    selected_features = selected_numeric + selected_categorical

    feature_preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), selected_numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), selected_categorical),
    ])
    feature_model = Pipeline([
        ("preprocess", feature_preprocess),
        ("model", LinearRegression()),
    ])
    feature_model.fit(X_fit[selected_features], y_fit)
    feature_validation_prediction = feature_model.predict(
        X_validation[selected_features]
    )
    feature_set_rows.append({
        "feature_set": feature_set_name,
        "feature_count": len(selected_features),
        "validation_MAE": mean_absolute_error(
            y_validation, feature_validation_prediction
        ),
    })

feature_set_comparison = pd.DataFrame(feature_set_rows)
without_hour_mae = feature_set_comparison.loc[
    feature_set_comparison["feature_set"].eq("without_hr"), "validation_MAE"
].iloc[0]
with_hour_mae = feature_set_comparison.loc[
    feature_set_comparison["feature_set"].eq("with_hr"), "validation_MAE"
].iloc[0]
print("[같은 train 내부 split·전처리·모형에서 hr 포함 여부만 비교]")
display(feature_set_comparison.round(3))
print(f"hr 추가 후 validation MAE 변화: {with_hour_mae - without_hour_mae:+.3f} 건/시간")


## 4. 예측값·오차·실험조건 검증

validation MAE 선택표에서 정한 최종 Pipeline의 예측값을 실제값과 나란히 보고, 그 Pipeline의 예측과 test 실제값이 같은 행 순서로 정렬됐는지 확인합니다.


In [ ]:
# 기본 후보와 선택 실험을 한 표에 모읍니다. test 점수는 아직 사용하지 않습니다.
candidate_rows = []
for row in validation_table.itertuples(index=False):
    if row.model == "mean baseline":
        candidate_id = "mean baseline"
    elif row.model == "linear regression":
        candidate_id = "linear regression with_hr"
    else:
        candidate_id = "decision tree depth 8"
    candidate_rows.append({
        "candidate_id": candidate_id,
        "validation_mae": row.validation_mae,
        "evidence": "기본 후보 비교",
    })

if "depth_comparison" in globals():
    for row in depth_comparison.itertuples(index=False):
        candidate_rows.append({
            "candidate_id": f"decision tree depth {int(row.max_depth)}",
            "validation_mae": row.validation_MAE,
            "evidence": "depth 비교",
        })

if "feature_set_comparison" in globals():
    for row in feature_set_comparison.itertuples(index=False):
        candidate_rows.append({
            "candidate_id": f"linear regression {row.feature_set}",
            "validation_mae": row.validation_MAE,
            "evidence": "hr 포함 여부 비교",
        })

extended_selection_table = (
    pd.DataFrame(candidate_rows)
    .sort_values("validation_mae")
    .drop_duplicates("candidate_id", keep="first")
    .reset_index(drop=True)
)
final_candidate_id = extended_selection_table.loc[0, "candidate_id"]

final_feature_columns = list(feature_columns)
if final_candidate_id == "mean baseline":
    final_model = DummyRegressor(strategy="mean")
elif final_candidate_id.startswith("linear regression"):
    use_hour = not final_candidate_id.endswith("without_hr")
    final_numeric = list(numeric_features)
    final_categorical = [
        column for column in categorical_features
        if use_hour or column != "hr"
    ]
    final_feature_columns = final_numeric + final_categorical
    final_preprocess = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scale", StandardScaler()),
        ]), final_numeric),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), final_categorical),
    ])
    final_model = Pipeline([
        ("preprocess", final_preprocess),
        ("model", LinearRegression()),
    ])
else:
    final_depth = int(final_candidate_id.rsplit(" ", 1)[-1])
    final_model = Pipeline([
        ("preprocess", clone(preprocess)),
        ("model", DecisionTreeRegressor(
            max_depth=final_depth,
            min_samples_leaf=10,
            random_state=42,
        )),
    ])

print("[test를 보기 전에 확정한 전체 validation 후보표]")
display(extended_selection_table.round(3))
print("최종 선택 후보:", final_candidate_id)

final_model.fit(X_train[final_feature_columns], y_train)
final_test_prediction = final_model.predict(X_test[final_feature_columns])
final_test_table = pd.DataFrame([{
    "selected_candidate": final_candidate_id,
    "feature_count": len(final_feature_columns),
    "test_mae": mean_absolute_error(y_test, final_test_prediction),
    "test_rows": len(y_test),
}])
print("[모든 선택을 끝낸 뒤 한 번 계산한 최종 test MAE]")
display(final_test_table.round(3))

examples = pd.DataFrame({
    "observed": y_test.to_numpy()[:8],
    "predicted": final_test_prediction[:8],
}, index=y_test.index[:8])
print("[최종 test의 실제값과 예측값 예시]")
display(examples.round(3))

checks = pd.Series({
    "target excluded from X": target_name not in X.columns,
    "fit and validation disjoint": X_fit.index.intersection(X_validation.index).empty,
    "train and test disjoint": X_train.index.intersection(X_test.index).empty,
    "three base validation candidates": len(validation_table) == 3,
    "finite selection MAE": np.isfinite(extended_selection_table["validation_mae"]).all(),
    "one selected candidate evaluated on test": len(final_test_table) == 1,
    "test prediction aligned": len(final_test_prediction) == len(y_test),
    "finite final test MAE": np.isfinite(final_test_table["test_mae"]).all(),
}, name="passed")
display(checks.to_frame())
assert checks.all()
print("예측 Pipeline 검증: PASS")
print("주의: test 결과를 본 뒤 feature·모형·깊이를 바꾸면 새 외부 test 또는 이후 기간 자료로 다시 평가해야 합니다.")


## 결과 기록과 ChatGPT 검토

다음 셀은 전체 validation 후보표, 최종 test 보고표와 검증표를 직접 넣습니다.

- **관찰:** 기준모형 대비 validation MAE와 최종 test MAE
- **조건:** 입력·제외열, fit·validation·test 역할, 선택된 feature·모형·깊이
- **한계:** 무작위 holdout과 test 확인 뒤 변경 금지 원칙


In [ ]:
review_prompt = f"""
Bike Sharing 예측 Pipeline을 다음 실제 출력만 사용해 검토해 주세요.

- target: {target_name}
- 입력 feature: {feature_columns}
- 결과누수 때문에 제외한 열: casual, registered
- fit/validation/test 행 수: {len(X_fit)}/{len(X_validation)}/{len(X_test)}
- test를 보기 전에 확정한 validation 후보표:
{extended_selection_table.to_string(index=False)}
- 최종 test 보고표:
{final_test_table.to_string(index=False)}
- 필수 검사:
{checks.to_string()}

관찰 | 분석조건 | 해석한계의 세 항목으로 답해 주세요. 선택 후보와 validation
MAE, 최종 test MAE·행 수를 실제 값으로 인용하고, test 결과를 본 뒤 feature·
모형·깊이를 바꾸려면 새 외부 test 또는 이후 기간 자료가 필요하다고 명시해 주세요.
""".strip()
print("[ChatGPT에 복사할 교시별 검토 프롬프트]\n")
print(review_prompt)


## 완료 확인

마지막 출력이 `예측 Pipeline 검증: PASS`인지 확인해 주세요. 이 검사는 필요한 계산이 실행됐는지 확인할 뿐, 연구 주장의 타당성을 자동 판정하지 않습니다.


## 결과를 정리할 질문

- 예측대상, 입력변수와 제외열을 어떤 기준으로 정했습니까?
- 후보는 어느 validation 지표로 정했고, 최종 test에는 몇 개의 Pipeline을 적용했습니까?
- 무작위 holdout 결과를 미래예측 성능으로 해석하지 않기 위해 다음에 어떤 검증이 필요합니까?
